In [ ]:
from cntmosaic.datasets import load_age_distribution, load_template_patterns
from cntmosaic.sim import (
    Stratification,
    PopulationConstructor,
    ParticipantGenerator,
    MatrixGenerator,
    ContactGenerator,
    PopulationConstructor,
)
from cntmosaic.dataloader import (
    DataLoader,
    ParticipantData,
    ContactData,
    PopulationData,
    StratPropData,
)
from cntmosaic.models import HiBRCfine, BRCfine
from cntmosaic.models.priors import PSpline2D
from cntmosaic.analysis import ModelSummariserBRC, svi_to_inference_data
from cntmosaic.vis import plot_mosaic

from numpyro.infer import log_likelihood
from numpyro.infer.initialization import init_to_value
from numpyro.infer.autoguide import AutoNormal
from numpyro.infer.util import _predictive
from numpyro.handlers import substitute

from jax.random import PRNGKey
from arviz import loo, compare

import altair as alt

## Generating Synthetic Data

In [ ]:
df_ref_pop = load_age_distribution("United_States", 80)

strats = [
    Stratification(
        name="sex",
        n_strata=2,
        ref_age_dist=df_ref_pop["P"].values,
        labels=["M", "F"],
        seed=0,
    ),
    Stratification(
        name="educ",
        n_strata=3,
        ref_age_dist=df_ref_pop["P"].values,
        labels=["Low", "Mid", "High"],
        seed=1,
    ),
]

In [ ]:
# Generate populations
pc = PopulationConstructor(strats)
df_pop = pc.df_P
df_pop_prop = pc.df_Q

In [ ]:
# Generate participants
pg = ParticipantGenerator(pc, n_part=1500)
df_part = pg.generate(seed=0)

df_part.head(5)

In [ ]:
templates = load_template_patterns("United_States", smooth=True, max_age=80)

# Generate contact matrices
mg = MatrixGenerator(templates)
cint_matrices = mg.generate_partial(pc, 10, seed=0)

for key in cint_matrices:
    print(f"Contact matrix for stratification: {key}")

In [ ]:
chart_list = []
for key, values in cint_matrices.items():
    if "M_" in key:
        chart = plot_mosaic(values, title=str(key), zlabel="Intensity")
        chart_list.append(chart)

alt.hconcat(*chart_list)

In [ ]:
chart_list = []
for key, values in cint_matrices.items():
    if "F" in key:
        chart = plot_mosaic(values, title=str(key), zlabel="Intensity")
        chart_list.append(chart)

alt.hconcat(*chart_list)

Here, we are going to hack the cint_matrices such that we don't have to stratify by education level. Our model selection procedure should choose the model that doesn't stratify by education as the best model.

In [ ]:
# Hack the contact matrices to remove education stratification
cint_matrices["M_Low->All"] = cint_matrices["M_Mid->All"]
cint_matrices["M_High->All"] = cint_matrices["M_Mid->All"]

cint_matrices["F_Low->All"] = cint_matrices["F_Mid->All"]
cint_matrices["F_High->All"] = cint_matrices["F_Mid->All"]

In [ ]:
# Generate contacts
cg = ContactGenerator(df_part, cint_matrices, "poisson", random_effects=True)
df_cnt = cg.generate(seed=0)

## Modelling

### Most Complicated Model

In [ ]:
part_data = ParticipantData(df_part, "id", "age", strat_var_cols=["sex", "educ"])
cnt_data = ContactData(df_cnt, "id", "age_cnt")
pop_data = PopulationData(df_pop, "age", "P", strat_var_cols=["sex", "educ"])
strat_prop_data = StratPropData(
    df_pop_prop, "age", strat_var_cols=["sex", "educ"], prop_col="Q"
)

dataloader = DataLoader(part_data, cnt_data, pop_data, strat_prop_data)

In [ ]:
# Define the model
priors = {
    "rate": PSpline2D("global", M=40),
    "sex": PSpline2D("partial", M=40),
    "educ": PSpline2D("partial", M=40),
}
model = HiBRCfine(dataloader, priors, "negbin")
model.print_model_shape()

In [ ]:
# Fit the model
init_values = {"baseline": -model.log_P.mean()}
guide = AutoNormal(model.model, init_loc_fn=init_to_value(values=init_values))
model.run_inference_svi(PRNGKey(0), guide, num_steps=5_000)

In [ ]:
idata_full = svi_to_inference_data(model)

## Model 2: No Stratification by Education

Fit a model that only stratifies by sex.

In [ ]:
part_data_best = ParticipantData(df_part, "id", "age", strat_var_cols="sex")
cnt_data_best = ContactData(df_cnt, "id", "age_cnt")
pop_data_best = PopulationData(df_pop, "age", "P", strat_var_cols=["sex"])
strat_prop_data_best = StratPropData(
    df_pop_prop, "age", strat_var_cols=["sex"], prop_col="Q"
)

dataloader_best = DataLoader(
    part_data_best, cnt_data_best, pop_data_best, strat_prop_data_best
)

In [ ]:
priors = {"rate": PSpline2D("global", M=40), "sex": PSpline2D("partial", M=40)}
model_best = HiBRCfine(dataloader_best, priors, "negbin")
model_best.print_model_shape()

In [ ]:
init_values = {"baseline": -model_best.log_P.mean()}
guide = AutoNormal(model_best.model, init_loc_fn=init_to_value(values=init_values))
model_best.run_inference_svi(PRNGKey(0), guide, num_steps=5_000)

From the data loader of the most complicated model, we can extract the for each observation for sex. The `dataloader.model_data.strat_data` contains `ixs` which is a dictionary of the coding for each stratification variable. 

In [ ]:
flat_ix = dataloader.model_data.strat_data["ixs"]["sex"]
kwargs_dict = {
    "y": dataloader.model_data.base_data["y"],
    "aid": dataloader.model_data.base_data["aid"],
    "bid": dataloader.model_data.base_data["bid"],
    "flat_ix": flat_ix,
    "flat_pixs": flat_ix,
    "log_N": dataloader.model_data.base_data["log_N"],
    "log_S": dataloader.model_data.base_data["log_S"],
}

In [ ]:
idata_best = svi_to_inference_data(model_best, kwargs_dict=kwargs_dict)

## Model 3: No Stratification

In [ ]:
part_data_simple = ParticipantData(df_part, "id", "age")
cnt_data_simple = ContactData(df_cnt, "id", "age_cnt")
pop_data_simple = PopulationData(df_pop, "age", "P")
dataloader_simple = DataLoader(part_data_simple, cnt_data_simple, pop_data_simple)

In [ ]:
prior = {"rate": PSpline2D("global", M=40)}
model_simple = BRCfine(dataloader_simple, prior, "negbin")
model_simple.print_model_shape()

In [ ]:
init_values = {"baseline": -model_simple.log_P.mean()}
guide = AutoNormal(model_simple.model, init_loc_fn=init_to_value(values=init_values))
model_simple.run_inference_svi(PRNGKey(0), guide, num_steps=5_000)

In [ ]:
kwargs_dict = {
    "y": dataloader.model_data.base_data["y"],
    "aid": dataloader.model_data.base_data["aid"],
    "bid": dataloader.model_data.base_data["bid"],
    "log_N": dataloader.model_data.base_data["log_N"],
    "log_S": dataloader.model_data.base_data["log_S"],
}

idata_simple = svi_to_inference_data(model_simple, kwargs_dict=kwargs_dict)

In [ ]:
idatas = {
    "simple_model": idata_simple,
    "best_model": idata_best,
    "full_model": idata_full,
}

In [ ]:
compare(idatas, ic="loo", method="stacking")